In [22]:
from sys import displayhook

import networkx as nx
import matplotlib.pyplot as plt
import os
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
import numpy as np

csv_path = os.path.join("data", "export_api_influencers.csv")
df = pd.read_csv(csv_path)

# display(len(df['Destino']))
df_copy = df.copy()

df_copy = df_copy.dropna(subset=["Cuentas_Que_Sigue"])
# df_copy = df_copy['Destino'].dropna()

print(len(df_copy))

# df_copy['Cuentas_Que_Sigue'].isnull()
df_copy

71


,Usuario,Ubicacion,Fecha_Creacion,Cuentas_Que_Sigue
0,Bot_5,India,2023-05-06,AMATEUR_1
1,fan_6,Mexico,2023-07-14,"GARNACHO, MESSI"
2,Bot_11,India,2023-02-14,AMATEUR_1
3,Bot_1,India,2023-11-24,AMATEUR_1
4,Bot_36,India,2023-06-24,AMATEUR_1
...,...,...,...,...
69,Bot_24,India,2023-11-29,AMATEUR_1
71,fan_3,Mexico,2023-12-05,GARNACHO
72,Bot_2,India,2023-02-27,AMATEUR_1
73,Bot_15,India,2023-01-16,AMATEUR_1


In [23]:
df_copy['Usuario'] = df_copy['Usuario'].str.strip().str.upper()
df_copy = df_copy.drop_duplicates(subset="Usuario")
df_copy

,Usuario,Ubicacion,Fecha_Creacion,Cuentas_Que_Sigue
0,BOT_5,India,2023-05-06,AMATEUR_1
1,FAN_6,Mexico,2023-07-14,"GARNACHO, MESSI"
2,BOT_11,India,2023-02-14,AMATEUR_1
3,BOT_1,India,2023-11-24,AMATEUR_1
4,BOT_36,India,2023-06-24,AMATEUR_1
...,...,...,...,...
67,BOT_22,India,2023-10-15,AMATEUR_1
68,BOT_3,India,2023-01-13,AMATEUR_1
69,BOT_24,India,2023-11-29,AMATEUR_1
71,FAN_3,Mexico,2023-12-05,GARNACHO


In [24]:
df_copy["Destino"] = df_copy["Cuentas_Que_Sigue"].str.split(",")

extremos = df_copy[["Usuario", "Destino"]].explode("Destino")

extremos["Destino"] = extremos["Destino"].str.strip().str.upper()
extremos = extremos[extremos["Destino"] != ""]
extremos = extremos[extremos["Usuario"] != extremos["Destino"]]
extremos = extremos.reset_index(drop=True)

print(len(extremos))
extremos


97


/var/folders/6m/8nbgcqbd65b4hkb9fz351syw0000gn/T/ipykernel_27539/701242058.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_copy["Destino"] = df_copy["Cuentas_Que_Sigue"].str.split(",")


,Usuario,Destino
0,BOT_5,AMATEUR_1
1,FAN_6,GARNACHO
2,FAN_6,MESSI
3,BOT_11,AMATEUR_1
4,BOT_1,AMATEUR_1
...,...,...
92,BOT_22,AMATEUR_1
93,BOT_3,AMATEUR_1
94,BOT_24,AMATEUR_1
95,FAN_3,GARNACHO


In [27]:
G = nx.from_pandas_edgelist(extremos, source="Usuario", target="Destino",
                            create_using=nx.DiGraph)

in_cent = nx.in_degree_centrality(G)
top3 = sorted(in_cent.items(), key=lambda x: x[1], reverse=True)[:3]
for user, score in top3:
    print(f"{user}: {score:.4f}")

AMATEUR_1: 0.7231
GARNACHO: 0.2000
MESSI: 0.1538


In [28]:
pr = nx.pagerank(G)
top_pr = sorted(pr.items(), key=lambda x: x[1], reverse=True)[:5]
for user, score in top_pr:
    print(f"{user}: {score:.4f}")
print("AMATEUR_1:", pr["AMATEUR_1"], "| GARNACHO:", pr["GARNACHO"])

G_und = G.to_undirected()
clust = nx.clustering(G_und)
print("Clustering AMATEUR_1:", clust["AMATEUR_1"])
print("Clustering GARNACHO:", clust["GARNACHO"])

AMATEUR_1: 0.1969
DE_PAUL: 0.0883
GARNACHO: 0.0843
SUAREZ: 0.0794
OTAMENDI: 0.0686
AMATEUR_1: 0.1969406767931266 | GARNACHO: 0.08428793226154431
Clustering AMATEUR_1: 0
Clustering GARNACHO: 0.13333333333333333


Amateur 1 parece el lider porque concentra la mayoria de los seguidores, de los cuales la mayoria son bots.
En pagerank amateur obtiene un valor mayor al de garnacho, ya que se basa en la cantidad de seguidores, y a amateur 1 solamente lo siguen cuentas que no siguen a nadie mas mientras que a garnacho lo siguen cuentas que vienen desde otras cuentas.

Y el verdadero influencer en este caso seria garnacho, ya que aunque tiene menor cantidad de seguidors, su audiencia es organica, y que se encuentra interconectada. Y se prefiere patroncinar a garnacho antes que a amateur_1